# Valider une table de données avec pandas

Nkap Microfinance, entreprise qui n'existe pas, nous transmet l'extraction de son portefeuille de prêts au 30/09/2026, avec un bordereau qui annonce le nombre de prêts et les totaux.
Avant toute analyse, une seule question : **cette table est-elle valide pour l'usage prévu** (le suivi du portefeuille et du risque) ?

Méthode suivie dans ce notebook :
1. écrire le **contrat** de la table *avant* de regarder les données ;
2. **diagnostiquer** dans un ordre précis, sans rien modifier ;
3. produire un **rapport de validation** et **décider** ;
4. **corriger** ce qui peut l'être de façon certaine, en traçant chaque correction ;
5. **revalider**, puis livrer une table d'analyse et la liste des anomalies à renvoyer à la source.

> **Rappel important : ** La table contient des données personnelles (nom, téléphone, e-mail, date de naissance).
> On ne les copie pas hors du poste de travail, on ne les affiche pas plus que nécessaire, et on les retire de la table d'analyse. Je tiens à préciser ici que c'est une exigence éthique qui est consignées dans la plupart des lois statistiques. donc vous êtes passible d'une peine pénale en fonction des lois si votre travail porte préjudice à une institution (entreprise comme personne)

In [ ]:
import json

import pandas as pd

In [ ]:

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

DOSSIER = "C:/Users/kfkwc/Downloads/Seance_Validation_notebook_donnees/Seance_Validation/data/" 
FICHIER = DOSSIER + "prets_nkap.csv"
DATE_EXTRACTION = pd.Timestamp("2026-09-30")

with open(DOSSIER + "bordereau_source.json", encoding="utf-8") as f:
    BORDEREAU = json.load(f)
BORDEREAU

{'nb_prets': 600,
 'total_montant': 435653000,
 'total_encours': 108214700,
 'date_extraction': '30/09/2026'}

## 0. Le contrat de la table (écrit avant de regarder les données)
Ce que je veux qu'on retienne ici est que le contrat dit ce que la table **doit** être : colonnes, types, valeurs obligatoires, domaines, plages, formats, règles de cohérence.
Sans contrat, « valide » n'a pas de sens.

In [31]:
COLONNES_ATTENDUES = ["id_pret", "id_client", "nom_client", "sexe", "date_naissance", "telephone",
                      "email", "agence", "date_octroi", "duree_mois", "montant", "taux_annuel",
                      "mensualite", "date_echeance", "statut", "encours"]

OBLIGATOIRES = ["id_pret", "id_client", "agence", "date_octroi", "duree_mois", "montant",
                "taux_annuel", "statut", "encours"]
FACULTATIVES = ["sexe", "telephone", "email"]

ENTIERS = ["duree_mois", "montant", "mensualite", "encours"]      # entiers, sans séparateur
DECIMAUX = ["taux_annuel"]                                      # décimal, virgule (0,18)
DATES = ["date_naissance", "date_octroi", "date_echeance"]     # jj/mm/aaaa

DOMAINES = {
    "sexe": {"F", "M"},
    "statut": {"En cours", "Soldé", "En retard", "Contentieux"},
    "duree_mois": {6, 12, 18, 24, 36},
}
PLAGES = {"montant": (25_000, 10_000_000), "taux_annuel": (0.05, 0.30), "encours": (0, None)}
AGE_MIN, AGE_MAX = 18, 75

FORMATS = {
    "id_pret": r"^PR-\d{5}$",
    "id_client": r"^CL-\d{4}$",
    "telephone": r"^\+237 6\d{2}( \d{2}){3}$",
    "email": r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$",
}
VALEURS_MANQUANTES = {"", "-", "?", "N/A", "NA", "n/a", "null", "NULL", "None", "nan"}
DONNEES_PERSONNELLES = ["nom_client", "date_naissance", "telephone", "email"]

REFERENTIEL_AGENCES = pd.read_csv(DOSSIER + "agences.csv", sep=";", dtype=str)
REFERENTIEL_AGENCES

,code_agence,nom_agence,region
0,AG-01,Yaoundé Centre,Centre
1,AG-02,Yaoundé Mvog-Mbi,Centre
2,AG-03,Douala Akwa,Littoral
3,AG-04,Douala Bonabéri,Littoral
4,AG-05,Bafoussam,Ouest
5,AG-06,Garoua,Nord
6,AG-07,Bamenda,Nord-Ouest
7,AG-08,Bertoua,Est


## Les outils du diagnostic

Chaque contrôle passera par la fonction `controle()` : il compte les lignes en échec, les range dans le **rapport**, et garde le détail dans la liste des **anomalies**.
Un contrôle ne modifiera jamais la table de donnée.

In [32]:
RAPPORT, ANOMALIES = [], []

def controle(table, etape, regle, dimension, severite, echec, colonne=None):
    # Contrôle ligne à ligne 
    echec = echec.fillna(False).astype(bool)
    n = int(echec.sum())
    RAPPORT.append({"etape": etape, "regle": regle, "dimension": dimension, "severite": severite,
                    "nb_echecs": n, "taux_%": round(100 * n / max(len(table), 1), 2)})
    for i in table.index[echec]:
        valeur = table.at[i, colonne] if colonne in table.columns else ""
        if colonne in DONNEES_PERSONNELLES:
            valeur = "(masqué)"                       #  on ne recopie pas la donnée
        ANOMALIES.append({"ligne_fichier": i + 2, "id_pret": table.at[i, "id_pret"] if "id_pret" in table.columns else "",
                          "regle": regle, "severite": severite, "colonne": colonne, "valeur": str(valeur)})
    return n

def controle_table(etape, regle, dimension, severite, n, detail=""):
    # Contrôle qui porte sur la table entière (colonnes, totaux).
    RAPPORT.append({"etape": etape, "regle": regle, "dimension": dimension, "severite": severite,
                    "nb_echecs": int(n), "taux_%": None, "detail": detail})
    return n

def est_manquant(s):
    return s.isna() | s.astype(str).str.strip().isin(VALEURS_MANQUANTES)

def vers_entier(s):
    # Conversion d'une valeur en entier sans séparateur. Sinon NaN.
    if pd.api.types.is_numeric_dtype(s):
        return s
    return pd.to_numeric(s.where(~est_manquant(s)), errors="coerce")

def vers_decimal(s):
    if pd.api.types.is_numeric_dtype(s):
        return s
    return pd.to_numeric(s.where(~est_manquant(s)).str.replace(",", ".", regex=False), errors="coerce")

def vers_date(s):
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    return pd.to_datetime(s.where(~est_manquant(s)), format="%d/%m/%Y", errors="coerce")

## Étape 1. Le fichier se lit-il, et a-t-il la taille annoncée ?

**Règle importante** : on lit tout **en texte** (`dtype=str`) et sans laisser pandas deviner les valeurs manquantes (`keep_default_na=False`).
On voit ainsi les données telles qu'elles sont, et non telles que pandas les interprète. L'original n'est **jamais** modifié donc on travaille sur une copie.

In [33]:
df_brut = pd.read_csv(FICHIER, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df = df_brut.copy()                       # copie

print("Lignes lues      :", len(df))
print("Prêts annoncés   :", BORDEREAU["nb_prets"])
print("Colonnes lues    :", df.shape[1], "| attendues :", len(COLONNES_ATTENDUES))
controle_table(1, "Nombre de lignes = nombre annoncé", "Rapprochement", "ALERTE",
               len(df) != BORDEREAU["nb_prets"], f"{len(df)} lues pour {BORDEREAU['nb_prets']} annoncées")
df.head(3)

Lignes lues      : 609
Prêts annoncés   : 600
Colonnes lues    : 17 | attendues : 16


,id_pret,id_client,nom_client,sexe,date_naissance,telephone,email,agence,date_octroi,duree_mois,Montant,taux_annuel,mensualite,date_echeance,statut,encours,commentaire
0,PR-00001,CL-0042,NANA Hervé,M,28/11/1972,+237 699 56 30 00,herve.nana79@exemple.cm,AG-02,27/10/2025,6,708000,"0,18",124272,27/04/2026,Soldé,0,
1,PR-00002,CL-0371,MANGA Thierry,M,27/01/1974,+237 686 95 65 81,thierry.manga10@exemple.cm,AG-06,2023-07-21,24,724000,"0,15",35104,21/07/2025,Soldé,0,
2,PR-00003,CL-0300,BIYONG Fabrice,M,15/05/1987,+237 685 59 54 11,fabrice.biyong52@exemple.cm,AG-08,07/01/2026,12,2775000,"0,18",254412,07/01/2027,En cours,903900,


## Étape 2. La structure : les colonnes attendues sont-elles là ?

C'est le premier contrôle **bloquant**. sans les bonnes colonnes, aucun autre contrôle n'a de sens et aussi vous analyse pourrait ne plus être possible

In [34]:
manquantes = [c for c in COLONNES_ATTENDUES if c not in df.columns]
en_trop = [c for c in df.columns if c not in COLONNES_ATTENDUES]
print("Colonnes manquantes :", manquantes)
print("Colonnes en trop    :", en_trop)

#  On compare les noms nettoyés (espaces, casse), l'idée c'est eviter que des colonnes soient considérées 
# comme manquantes alors qu'elles sont présentes mais mal nommées
proches = {c: c.strip().lower() for c in en_trop if c.strip().lower() in COLONNES_ATTENDUES}
print("Noms à harmoniser   :", proches)

vides = [c for c in df.columns if est_manquant(df[c]).all()]
print("Colonnes entièrement vides :", vides)

controle_table(2, "Colonnes attendues présentes", "Structure", "BLOQUANT", len(manquantes), str(manquantes))
controle_table(2, "Pas de colonne en trop", "Structure", "ALERTE", len(en_trop), str(en_trop))
controle_table(2, "Pas de colonne entièrement vide", "Structure", "ALERTE", len(vides), str(vides))

Colonnes manquantes : ['montant']
Colonnes en trop    : ['Montant ', 'commentaire']
Noms à harmoniser   : {'Montant ': 'montant'}
Colonnes entièrement vides : ['commentaire']


1

**Décision à prendre** La structure conditionne tout le reste. on la corrige **tout de suite**, de façon certaine et tracée.
`"Montant "` est bien la colonne `montant` (même contenu, nom mal écrit) ; `commentaire` est vide et hors contrat.

In [35]:
JOURNAL = []
def journaliser(action, lignes_concernees, justification):
    JOURNAL.append({"action": action, "lignes": int(lignes_concernees), "justification": justification})

df = df.rename(columns=proches)
journaliser("Harmonisation des noms de colonnes", 0, f"{proches} : espaces et casse, contenu identique")
df = df.drop(columns=vides)
journaliser("Suppression des colonnes vides", 0, f"{vides} : entièrement vides, hors contrat")
print(list(df.columns) == COLONNES_ATTENDUES)

True


## Étape 3. Les lignes entièrement vides

In [36]:
lignes_vides = df.apply(est_manquant).all(axis=1)
controle(df, 3, "Pas de ligne entièrement vide", "Complétude", "ALERTE", lignes_vides)
print("Lignes vides aux lignes du fichier :", list(df.index[lignes_vides] + 2))

t = df[~lignes_vides]                     # les contrôles suivants portent sur les lignes non vides
len(t)

Lignes vides aux lignes du fichier : [41, 251, 386]


606

## Étape 4. Type de variable. Chaque valeur est-elle lisible selon le contrat ?

Une valeur **présente** mais **non convertible** est un échec de type. Les valeurs absentes seront comptées à l'étape 6.

In [37]:
def etape_types(t):
    for c in ENTIERS:
        n = controle(t, 4, f"{c} : entier lisible", "Validité (type)", "BLOQUANT",
                     ~est_manquant(t[c]) & vers_entier(t[c]).isna(), c)
    for c in DECIMAUX:
        controle(t, 4, f"{c} : décimal lisible", "Validité (type)", "BLOQUANT",
                 ~est_manquant(t[c]) & vers_decimal(t[c]).isna(), c)

etape_types(t)
pd.DataFrame(RAPPORT).query("etape == 4")

,etape,regle,dimension,severite,nb_echecs,taux_%,detail
5,4,duree_mois : entier lisible,Validité (type),BLOQUANT,0,0.00,NaN
6,4,montant : entier lisible,Validité (type),BLOQUANT,12,1.98,NaN
7,4,mensualite : entier lisible,Validité (type),BLOQUANT,0,0.00,NaN
8,4,encours : entier lisible,Validité (type),BLOQUANT,0,0.00,NaN
9,4,taux_annuel : décimal lisible,Validité (type),BLOQUANT,0,0.00,NaN


In [38]:
# Que contiennent les montants illisibles ? On regarde avant de décider.
illisibles = t.loc[~est_manquant(t["montant"]) & vers_entier(t["montant"]).isna(), "montant"]
illisibles.value_counts().head(8)

montant
565 000      1
314 000      1
218 000      1
5 000 000    1
801 000      1
86 000       1
400 000      1
710 000      1
Name: count, dtype: int64

## Étape 5. Les dates : au format jj/mm/aaaa et réelles

In [39]:
def etape_dates(t):
    for c in DATES:
        controle(t, 5, f"{c} : date jj/mm/aaaa valide", "Validité (type)", "BLOQUANT",
                 ~est_manquant(t[c]) & vers_date(t[c]).isna(), c)

etape_dates(t)
t.loc[~est_manquant(t["date_octroi"]) & vers_date(t["date_octroi"]).isna(), "date_octroi"].head(6)

1      2023-07-21
10     2025-09-02
29     2026-04-24
83     2025-01-24
84     2023-12-28
124    2026-08-22
Name: date_octroi, dtype: str

## Étape 6. La complétude : les valeurs obligatoires sont-elles présentes ?

On compte aussi les **pseudo-manquants** : `-`, `?`, `N/A`… qui ne sont pas vides mais ne disent rien.

In [40]:
def etape_completude(t):
    for c in OBLIGATOIRES:
        controle(t, 6, f"{c} renseigné", "Complétude", "BLOQUANT", est_manquant(t[c]), c)
    for c in FACULTATIVES:
        controle(t, 6, f"{c} renseigné", "Complétude", "ALERTE", est_manquant(t[c]), c)

etape_completude(t)
pd.DataFrame({"manquants": t.apply(est_manquant).sum(),
              "dont pseudo-manquants": t.apply(lambda s: s.astype(str).str.strip().isin(VALEURS_MANQUANTES - {""})).sum()}
            ).query("manquants > 0")

,manquants,dont pseudo-manquants
sexe,8,2
telephone,21,0
email,35,0
agence,4,4
montant,2,2


## Étape 7. L'unicité : une ligne par prêt

Deux cas très différents : le **doublon exact** (la même ligne deux fois, sans ambiguïté) et le **conflit de clé** (le même identifiant pour deux contenus différents : lequel est le bon ?).

In [41]:
def etape_unicite(t):
    doublon_exact = t.duplicated(keep="first")
    controle(t, 7, "Pas de ligne en double", "Unicité", "ALERTE", doublon_exact, "id_pret")
    sans_doublons = t[~doublon_exact]
    conflit = sans_doublons.duplicated("id_pret", keep=False)
    controle(t, 7, "id_pret unique", "Unicité", "BLOQUANT",
             pd.Series(t.index.isin(sans_doublons.index[conflit]), index=t.index), "id_pret")

etape_unicite(t)
conflits = t[~t.duplicated()]
conflits[conflits.duplicated("id_pret", keep=False)].sort_values("id_pret")[["id_pret", "id_client", "montant", "mensualite"]]

,id_pret,id_client,montant,mensualite
87,PR-00543,CL-0445,815000,50461
550,PR-00543,CL-0445,565000,34982
153,PR-00600,CL-0199,2402000,222508
608,PR-00600,CL-0199,2152000,199349


## Étape 8. Les domaines : la valeur fait-elle partie des valeurs autorisées ?

In [42]:
def etape_domaines(t):
    for c, autorises in DOMAINES.items():
        valeurs = vers_entier(t[c]) if c in ENTIERS else t[c]
        sev = "ALERTE" if c in FACULTATIVES else "BLOQUANT"
        controle(t, 8, f"{c} dans le domaine autorisé", "Validité (domaine)", sev,
                 ~est_manquant(t[c]) & valeurs.notna() & ~valeurs.isin(autorises), c)

etape_domaines(t)
for c in ["statut", "sexe"]:
    print(c, ":", dict(t.loc[~est_manquant(t[c]) & ~t[c].isin(DOMAINES[c]), c].value_counts()))

statut : {'en cours': np.int64(4), 'EN COURS ': np.int64(3), 'Solde': np.int64(2)}
sexe : {'Femme': np.int64(3)}


## Étape 9. la valeur est-elle plausible ? 

In [43]:
def age_a_l_octroi(t):
    return (vers_date(t["date_octroi"]) - vers_date(t["date_naissance"])).dt.days / 365.25

def etape_plages(t):
    for c, (mini, maxi) in PLAGES.items():
        valeurs = vers_decimal(t[c]) if c in DECIMAUX else vers_entier(t[c])
        hors = (valeurs < mini) if mini is not None else False
        if maxi is not None:
            hors = hors | (valeurs > maxi)
        fmt = lambda x: f"{x:,}".replace(",", " ") if float(x).is_integer() else str(x)
        libelle = f"{c} entre {fmt(mini)} et {fmt(maxi)}" if maxi is not None else f"{c} >= {fmt(mini)}"
        controle(t, 9, libelle, "Validité (plage)", "BLOQUANT", hors & valeurs.notna(), c)
    age = age_a_l_octroi(t)
    controle(t, 9, f"âge à l'octroi entre {AGE_MIN} et {AGE_MAX} ans", "Validité (plage)", "BLOQUANT",
             age.notna() & ((age < AGE_MIN) | (age > AGE_MAX)), "date_naissance")

etape_plages(t)
pd.DataFrame(RAPPORT).query("etape == 9")[["regle", "nb_echecs"]]

,regle,nb_echecs
30,montant entre 25 000 et 10 000 000,3
31,taux_annuel entre 0.05 et 0.3,4
32,encours >= 0,0
33,âge à l'octroi entre 18 et 75 ans,2


## Étape 10. Les formats : identifiants, téléphone, e-mail

In [44]:
def etape_formats(t):
    for c, motif in FORMATS.items():
        sev = "ALERTE" if c in FACULTATIVES else "BLOQUANT"
        controle(t, 10, f"{c} au format attendu", "Validité (format)", sev,
                 ~est_manquant(t[c]) & ~t[c].astype(str).str.match(motif), c)

etape_formats(t)
pd.DataFrame(RAPPORT).query("etape == 10")[["regle", "severite", "nb_echecs"]]

,regle,severite,nb_echecs
34,id_pret au format attendu,BLOQUANT,2
35,id_client au format attendu,BLOQUANT,0
36,telephone au format attendu,ALERTE,7
37,email au format attendu,ALERTE,5


## Étape 11. La cohérence à l'intérieur de chaque ligne

Chaque valeur peut être valide seule, et la ligne incohérente : une échéance qui ne correspond pas à la durée, un prêt soldé qui a encore un encours…

In [45]:
def echeance_calculee(t):
    octroi, duree = vers_date(t["date_octroi"]), vers_entier(t["duree_mois"])
    return pd.Series([o + pd.DateOffset(months=int(d)) if pd.notna(o) and pd.notna(d) else pd.NaT
                      for o, d in zip(octroi, duree)], index=t.index)

def mensualite_calculee(t):
    p, n, r = vers_entier(t["montant"]), vers_entier(t["duree_mois"]), vers_decimal(t["taux_annuel"]) / 12
    return p * r / (1 - (1 + r) ** -n)

def etape_coherence(t):
    ech = vers_date(t["date_echeance"])
    calc = echeance_calculee(t)
    controle(t, 11, "échéance = octroi + durée", "Cohérence", "BLOQUANT",
             ech.notna() & calc.notna() & (ech != calc), "date_echeance")
    encours, montant = vers_entier(t["encours"]), vers_entier(t["montant"])
    controle(t, 11, "prêt soldé => encours nul", "Cohérence", "BLOQUANT",
             (t["statut"] == "Soldé") & (encours > 0), "encours")
    controle(t, 11, "encours <= montant", "Cohérence", "BLOQUANT", encours > montant, "encours")
    ecart = (vers_entier(t["mensualite"]) - mensualite_calculee(t)).abs() / mensualite_calculee(t)
    controle(t, 11, "mensualité conforme au calcul (1 %)", "Cohérence", "ALERTE", ecart > 0.01, "mensualite")

etape_coherence(t)
pd.DataFrame(RAPPORT).query("etape == 11")[["regle", "severite", "nb_echecs"]]

,regle,severite,nb_echecs
38,échéance = octroi + durée,BLOQUANT,6
39,prêt soldé => encours nul,BLOQUANT,6
40,encours <= montant,BLOQUANT,5
41,mensualité conforme au calcul (1 %),ALERTE,10


## Étape 12. La cohérence avec les autres tables : l'intégrité référentielle

In [46]:
def etape_referentiel(t):
    connues = set(REFERENTIEL_AGENCES["code_agence"])
    controle(t, 12, "agence présente dans le référentiel", "Cohérence (référentielle)", "BLOQUANT",
             ~est_manquant(t["agence"]) & ~t["agence"].isin(connues), "agence")

etape_referentiel(t)
t.loc[~est_manquant(t["agence"]) & ~t["agence"].isin(REFERENTIEL_AGENCES["code_agence"]), "agence"].value_counts()

agence
AG-09    4
Name: count, dtype: int64

## Étape 13. Les valeurs aberrantes : suspectes, pas forcément fausses

Une valeur hors de l'ordinaire n'est **pas** une erreur. c'est une **alerte** à examiner avec le métier.
Méthode simple et robuste : l'écart interquartile (IQR). On signale au-delà de 3 IQR. De beaux graphiques permettent d'en suspecter l'existence

In [47]:
def etape_aberrantes(t):
    m = vers_entier(t["montant"])
    m_plausible = m[(m > 0)]
    q1, q3 = m_plausible.quantile([0.25, 0.75])
    haut = q3 + 3 * (q3 - q1)
    controle(t, 13, f"montant sous le seuil d'alerte ({haut:,.0f})", "Vraisemblance", "ALERTE", m > haut, "montant")
    return haut

seuil = etape_aberrantes(t)
print(f"Seuil d'alerte : {seuil:,.0f} FCFA")
t.loc[vers_entier(t["montant"]) > seuil, ["id_pret", "montant", "agence"]].sort_values("montant")

Seuil d'alerte : 2,620,000 FCFA


,id_pret,montant,agence
581,PR-00573,2696000,AG-04
385,PR-00378,2705000,AG-03
2,PR-00003,2775000,AG-08
370,PR-00364,2776000,AG-02
17,PR-00018,3047000,AG-01
206,PR-00206,3233000,AG-03
209,PR-00206,3233000,AG-03
588,PR-00580,3237000,AG-05
336,PR-00330,3332000,AG-02
313,PR-00307,3411000,AG-08


## Étape 14. L'actualité : les dates sont-elles cohérentes avec la date d'extraction ?

In [48]:
def etape_actualite(t):
    octroi = vers_date(t["date_octroi"])
    controle(t, 14, "date d'octroi <= date d'extraction", "Actualité", "BLOQUANT", octroi > DATE_EXTRACTION, "date_octroi")
    return octroi.max()

print("Dernière date d'octroi lue :", etape_actualite(t))

Dernière date d'octroi lue : 2026-11-15 00:00:00


## Étape 15. Le rapprochement avec la source

Le contrôle ici, détecte ce que tous les autres laissent passer. **les totaux concordent-ils avec ce que la source annonce ?**
On compare sur les prêts distincts (sans doublons exacts).

In [49]:
def etape_rapprochement(t):
    u = t[~t.duplicated()]
    nb = u["id_pret"].nunique()
    total = vers_entier(u.drop_duplicates("id_pret")["montant"]).sum()
    controle_table(15, "nombre de prêts distincts = bordereau", "Exactitude (rapprochement)", "BLOQUANT",
                   nb != BORDEREAU["nb_prets"], f"{nb} pour {BORDEREAU['nb_prets']}")
    controle_table(15, "total des montants = bordereau", "Exactitude (rapprochement)", "BLOQUANT",
                   round(total) != BORDEREAU["total_montant"],
                   f"{total:,.0f} pour {BORDEREAU['total_montant']:,.0f} (écart {total - BORDEREAU['total_montant']:,.0f})")
    return nb, total

nb, total = etape_rapprochement(t)
print(f"Prêts distincts : {nb}  |  total lisible : {total:,.0f}  |  bordereau : {BORDEREAU['total_montant']:,.0f}")

Prêts distincts : 600  |  total lisible : 517,362,000  |  bordereau : 435,653,000


## Le rapport de validation

Une table est **valide** quand toutes les règles **bloquantes** sont à zéro, et que les **alertes** ont été examinées.

In [50]:
rapport = pd.DataFrame(RAPPORT)
rapport_echecs = rapport[rapport["nb_echecs"] > 0]
print("Règles contrôlées :", len(rapport), "| en échec :", len(rapport_echecs))
print("Bloquantes en échec :", int((rapport_echecs["severite"] == "BLOQUANT").sum()),
      "| alertes :", int((rapport_echecs["severite"] == "ALERTE").sum()))
rapport_echecs[["etape", "regle", "dimension", "severite", "nb_echecs", "taux_%"]]

Règles contrôlées : 47 | en échec : 31
Bloquantes en échec : 18 | alertes : 13


,etape,regle,dimension,severite,nb_echecs,taux_%
0,1,Nombre de lignes = nombre annoncé,Rapprochement,ALERTE,1,NaN
1,2,Colonnes attendues présentes,Structure,BLOQUANT,1,NaN
2,2,Pas de colonne en trop,Structure,ALERTE,2,NaN
3,2,Pas de colonne entièrement vide,Structure,ALERTE,1,NaN
4,3,Pas de ligne entièrement vide,Complétude,ALERTE,3,0.49
6,4,montant : entier lisible,Validité (type),BLOQUANT,12,1.98
11,5,date_octroi : date jj/mm/aaaa valide,Validité (type),BLOQUANT,17,2.81
15,6,agence renseigné,Complétude,BLOQUANT,4,0.66
18,6,montant renseigné,Complétude,BLOQUANT,2,0.33
22,6,sexe renseigné,Complétude,ALERTE,8,1.32


In [51]:
anomalies = pd.DataFrame(ANOMALIES)
lignes_bloquees = anomalies.loc[anomalies["severite"] == "BLOQUANT", "ligne_fichier"].nunique()
print("Anomalies relevées :", len(anomalies))
print("Lignes avec au moins une anomalie bloquante :", lignes_bloquees, "sur", len(t))
DECISION = "NON VALIDE en l'état" if (rapport_echecs["severite"] == "BLOQUANT").any() else "VALIDE"
print("Décision :", DECISION)

Anomalies relevées : 195
Lignes avec au moins une anomalie bloquante : 81 sur 606
Décision : NON VALIDE en l'état


## Corriger la table : on traite seulement ce qui est certain, et toujours tracé

| Type de défaut | Ce qu'on fait |
|---|---|
| Correction **certaine** (forme, pas fond) | On corrige, on trace dans le journal |
| Valeur **fausse ou ambiguë** | On ne devine pas : on marque la ligne et on renvoie à la source |
| **Alerte** | On examine ; on garde la ligne, marquée « à vérifier » |

Il est important de savoir dans la pratique il ya beaucoup d'arbitrages qu'on fait pour gérer certaines erreurs (en fonction des méthodes parfois il faudra diagnostiquer l'impact des valeurs suspectes par exemple, l'impact de l'exclusion d'une ligne de la base, etc.)

In [52]:
c = t.copy()

n = int(c.duplicated().sum())
c = c[~c.duplicated()]
journaliser("Suppression des doublons exacts", n, "lignes strictement identiques : aucune information perdue")
journaliser("Exclusion des lignes vides", int(lignes_vides.sum()), "aucune donnée")

a_corriger = ~est_manquant(c["montant"]) & vers_entier(c["montant"]).isna() & c["montant"].str.fullmatch(r"-?[\d ]+")
c.loc[a_corriger, "montant"] = c.loc[a_corriger, "montant"].str.replace(" ", "", regex=False)
journaliser("montant : retrait des espaces de milliers", a_corriger.sum(), "« 1 250 000 » vaut 1250000")

iso = ~est_manquant(c["date_octroi"]) & vers_date(c["date_octroi"]).isna() & c["date_octroi"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
c.loc[iso, "date_octroi"] = pd.to_datetime(c.loc[iso, "date_octroi"], format="%Y-%m-%d").dt.strftime("%d/%m/%Y")
journaliser("date_octroi : format aaaa-mm-jj converti", iso.sum(), "même date, autre écriture")

STATUTS = {"en cours": "En cours", "soldé": "Soldé", "solde": "Soldé", "en retard": "En retard", "contentieux": "Contentieux"}
norm = c["statut"].str.strip().str.lower().map(STATUTS)
modif = norm.notna() & (norm != c["statut"])
c.loc[modif, "statut"] = norm[modif]
journaliser("statut : casse et espaces harmonisés", modif.sum(), "« EN COURS » vaut « En cours »")

SEXES = {"femme": "F", "f": "F", "homme": "M", "m": "M"}
s_norm = c["sexe"].str.strip().str.lower().map(SEXES)
modif = s_norm.notna() & (s_norm != c["sexe"])
c.loc[modif, "sexe"] = s_norm[modif]
journaliser("sexe : libellés harmonisés", modif.sum(), "« Femme » vaut « F »")

for col in ["agence", "sexe"]:
    pseudo = c[col].isin(VALEURS_MANQUANTES - {""})
    c.loc[pseudo, col] = ""
    journaliser(f"{col} : pseudo-manquants rendus explicites", pseudo.sum(), "« - » et « ? » ne sont pas des valeurs")

journal = pd.DataFrame(JOURNAL)
journal

,action,lignes,justification
0,Harmonisation des noms de colonnes,0,"{'Montant ': 'montant'} : espaces et casse, co..."
1,Suppression des colonnes vides,0,"['commentaire'] : entièrement vides, hors contrat"
2,Suppression des doublons exacts,4,lignes strictement identiques : aucune informa...
3,Exclusion des lignes vides,3,aucune donnée
4,montant : retrait des espaces de milliers,12,« 1 250 000 » vaut 1250000
5,date_octroi : format aaaa-mm-jj converti,15,"même date, autre écriture"
6,statut : casse et espaces harmonisés,9,« EN COURS » vaut « En cours »
7,sexe : libellés harmonisés,3,« Femme » vaut « F »
8,agence : pseudo-manquants rendus explicites,3,« - » et « ? » ne sont pas des valeurs
9,sexe : pseudo-manquants rendus explicites,2,« - » et « ? » ne sont pas des valeurs


## Revalider : la même batterie de contrôles, sur la table corrigée

on **relance tous les contrôles** pour se rassurer que la correction a fiabiliser la table. 

In [53]:
def valider(table):
    RAPPORT.clear(); ANOMALIES.clear()
    etape_types(table); etape_dates(table); etape_completude(table); etape_unicite(table)
    etape_domaines(table); etape_plages(table); etape_formats(table); etape_coherence(table)
    etape_referentiel(table); etape_aberrantes(table); etape_actualite(table); etape_rapprochement(table)
    return pd.DataFrame(RAPPORT), pd.DataFrame(ANOMALIES)

rapport2, anomalies2 = valider(c)
r2 = rapport2[rapport2["nb_echecs"] > 0]
print("Règles en échec après correction :", len(r2), "(avant :", len(rapport_echecs), ")")
r2[["etape", "regle", "severite", "nb_echecs"]]

Règles en échec après correction : 22 (avant : 31 )


,etape,regle,severite,nb_echecs
6,5,date_octroi : date jj/mm/aaaa valide,BLOQUANT,2
10,6,agence renseigné,BLOQUANT,3
13,6,montant renseigné,BLOQUANT,2
17,6,sexe renseigné,ALERTE,8
18,6,telephone renseigné,ALERTE,20
19,6,email renseigné,ALERTE,35
21,7,id_pret unique,BLOQUANT,4
24,8,duree_mois dans le domaine autorisé,BLOQUANT,2
25,9,montant entre 25 000 et 10 000 000,BLOQUANT,3
26,9,taux_annuel entre 0.05 et 0.3,BLOQUANT,4


## Marquer chaque ligne, puis livrer

Chaque ligne reçoit un **statut de validation**. `rejetée` (au moins une anomalie bloquante), `à vérifier` (seulement des alertes) ou `valide`.
On ne supprime rien en silence.

In [54]:
bloquantes = set(anomalies2.loc[anomalies2["severite"] == "BLOQUANT", "ligne_fichier"])
alertes = set(anomalies2.loc[anomalies2["severite"] == "ALERTE", "ligne_fichier"])
ligne = c.index + 2
c["statut_validation"] = "valide"
c.loc[ligne.isin(alertes), "statut_validation"] = "à vérifier"
c.loc[ligne.isin(bloquantes), "statut_validation"] = "rejetée"
c["statut_validation"].value_counts()

statut_validation
valide        465
à vérifier     92
rejetée        45
Name: count, dtype: int64

In [55]:
# Table d'analyse : lignes non rejetées, typées, et SANS données personnelles directes
analyse = c[c["statut_validation"] != "rejetée"].copy()
for col in ENTIERS:
    analyse[col] = vers_entier(analyse[col]).astype("Int64")
analyse["taux_annuel"] = vers_decimal(analyse["taux_annuel"])
for col in DATES:
    analyse[col] = vers_date(analyse[col])
analyse["age_octroi"] = age_a_l_octroi(analyse).round(0).astype("Int64")
analyse = analyse.drop(columns=["nom_client", "telephone", "email", "date_naissance"])

rejet = c[c["statut_validation"] == "rejetée"]
print("Prêts retenus pour l'analyse :", len(analyse), "| rejetés :", len(rejet))
print(f"Montant retenu : {analyse['montant'].sum():,.0f} FCFA")
analyse.dtypes

Prêts retenus pour l'analyse : 557 | rejetés : 45
Montant retenu : 405,948,000 FCFA


id_pret                         str
id_client                       str
sexe                            str
agence                          str
date_octroi          datetime64[us]
duree_mois                    Int64
montant                       Int64
taux_annuel                 float64
mensualite                    Int64
date_echeance        datetime64[us]
statut                          str
encours                       Int64
statut_validation               str
age_octroi                    Int64
dtype: object

In [56]:
anomalies2.to_csv("anomalies_a_transmettre.csv", sep=";", index=False, encoding="utf-8")   # données perso masquées
journal.to_csv("journal_corrections.csv", sep=";", index=False, encoding="utf-8")
rapport2.to_csv("rapport_validation.csv", sep=";", index=False, encoding="utf-8")
analyse.to_csv("prets_analyse.csv", sep=";", index=False, encoding="utf-8")
print("Fichiers écrits : anomalies_a_transmettre, journal_corrections, rapport_validation, prets_analyse")

Fichiers écrits : anomalies_a_transmettre, journal_corrections, rapport_validation, prets_analyse
